In [1]:
import glob, os, shutil, zipfile
dst = "/kaggle/working/unlearning-project"
shutil.rmtree(dst, ignore_errors=True)
src = None
for p in glob.glob("/kaggle/input/**/run_all.sh", recursive=True):
    if "UL_BS" in open(p).read():
        src = os.path.dirname(p)
if src is None:
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        zipfile.ZipFile(z).extractall("/tmp/unz")
    for p in glob.glob("/tmp/unz/**/run_all.sh", recursive=True):
        if "UL_BS" in open(p).read():
            src = os.path.dirname(p)
assert src, "v2 code nahi mila - v2 zip upload karein"
shutil.copytree(src, dst)
os.chdir(dst)
!pip install -q rouge-score bitsandbytes
!nvidia-smi --query-gpu=name --format=csv
print("v2 code OK:", sorted(os.listdir()))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 50.4 MB/s eta 0:00:00
name
Tesla T4
Tesla T4
v2 code OK: ['README.md', 'aggregate.py', 'common.py', 'download_data.py', 'evaluate.py', 'references.bib', 'requirements.txt', 'run_all.sh', 'tests', 'train_ft.py', 'unlearn.py']


In [2]:
%cd /kaggle/working/unlearning-project
!rm -rf ckpt && mkdir -p /tmp/ckpt && ln -s /tmp/ckpt ckpt
!bash run_all.sh

/kaggle/working
[20:17:06] GPUs: 2   model: Qwen/Qwen2.5-0.5B-Instruct   forget: forget10
[20:17:33] === phase 1: Model A + Oracle (retrain without forget set)
[20:17:33] START  A_original  (GPU 0)
[20:17:33] START  B0_retrain  (GPU 1)
[21:02:06] DONE   B0_retrain
[21:08:12] DONE   A_original
[21:08:13] kept A_original -> /kaggle/working/keep_ckpt
[21:08:14] kept B0_retrain -> /kaggle/working/keep_ckpt
[21:08:14] === phase 2: unlearning
[21:08:14] START  B_ft_retain  (GPU 0)
[21:08:14] START  eval_B0_retrain  (GPU 1)
[21:16:13] DONE   eval_B0_retrain
[21:16:13] START  D_npo_rt  (GPU 1)
[21:17:42] DONE   B_ft_retain
[21:17:42] START  C_graddiff  (GPU 0)
[21:28:14] DONE   C_graddiff
[21:28:14] START  E_ga  (GPU 0)
[21:28:18] DONE   D_npo_rt
[21:28:18] START  F_simnpo  (GPU 1)
[21:34:09] DONE   E_ga
[21:39:02] DONE   F_simnpo
[21:39:02] === phase 3: evaluation
[21:39:02] START  eval_A_original  (GPU 0)
[21:39:02] START  eval_D_npo_rt  (GPU 1)
[21:46:55] DONE   eval_A_original
[21:46:55] S

In [3]:
%cd /kaggle/working/unlearning-project
!zip -qr /kaggle/working/results_for_analysis.zip results logs
!ls -la /kaggle/working

/kaggle/working
total 596
drwxr-xr-x 4 root root   4096 Oct 10 22:24 .
drwxr-xr-x 8 root root   4096 Oct 10 20:16 ..
drwxr-xr-x 4 root root   4096 Oct 10 21:08 keep_ckpt
-rw-r--r-- 1 root root  14646 Oct 10 22:24 __notebook__.ipynb
-rw-r--r-- 1 root root 575373 Oct 10 22:24 results_for_analysis.zip
drwxr-xr-x 7 root root   4096 Oct 10 20:17 unlearning-project
